In [1]:
%cd ../../../

/path/to/repo/my-coles-gnn-experimetns/scenario_age_pred


/path/to/repo/.local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
from collections import defaultdict
import os

BATCH_SIZE = 40
MAX_EPOCHES_OPTIONS = range(10, 160, 10) 
SPLIT_COUNT = 2


MODEL_EPOCH_MAP = {
    "wl-0.5_gnn-GAT_residual-True_weight_decay-0": [31, 91], 
    "wl-0.5_gnn-GraphSAGE_residual-True_weight_decay-0": [31, 71],
    "GRACE-gnn-GAT-weight_decay-0.00001__residual-true__num_layers_3__emb_size_128": [11]
}




existing_experiments = []

for gnn_subexperiment_name, gnn_pretrain_epochs_options in MODEL_EPOCH_MAP.items():
    for pretrain_epoch in gnn_pretrain_epochs_options:
        for train_epochs in MAX_EPOCHES_OPTIONS:
            experiment_name=f"coles_only__item_id_embed_init_with_pretrained_gnn__bs_{BATCH_SIZE}__split_count_{SPLIT_COUNT}__{gnn_subexperiment_name}__pretrain_epoches_{pretrain_epoch}__epoches_{train_epochs}"
            if os.path.exists(f'data/{experiment_name}_embeddings.pickle'):
                existing_experiments.append(experiment_name)
            experiment_name_v2=f"coles_only__item_id_embed_init_with_pretrained_gnn__bs_{BATCH_SIZE}__split_count_{SPLIT_COUNT}__{gnn_subexperiment_name}__pretrain_epoches_{pretrain_epoch}__{train_epochs}_epoches"
            if os.path.exists(f'data/{experiment_name_v2}_embeddings.pickle'):
                existing_experiments.append(experiment_name_v2)


In [3]:
existing_experiments

['coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_31__10_epoches',
 'coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_31__20_epoches',
 'coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_31__30_epoches',
 'coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_31__40_epoches',
 'coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_31__50_epoches',
 'coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_31__60_epoches',
 'coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_cou

In [4]:
config_item_template = """  {experiment_name}:
    enabled: true
    read_params: 
      file_name: ${{hydra:runtime.cwd}}/data/{experiment_name}_embeddings.pickle
    target_options: {{}}
"""

for experiment in existing_experiments:
    print(config_item_template.format(experiment_name=experiment))

  coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_31__10_epoches:
    enabled: true
    read_params: 
      file_name: ${hydra:runtime.cwd}/data/coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_31__10_epoches_embeddings.pickle
    target_options: {}

  coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_31__20_epoches:
    enabled: true
    read_params: 
      file_name: ${hydra:runtime.cwd}/data/coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_31__20_epoches_embeddings.pickle
    target_options: {}

  coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_31__30_epo

In [5]:
from omegaconf import OmegaConf 

In [6]:
import sys
import os

sys.path.append(os.path.dirname(os.getcwd()))

In [7]:
from ptls_extension_2024_research.latex_table_creation.latex_table_creation import create_latex_table, get_metrics
from ptls_extension_2024_research.latex_table_creation.experiment_dicts_list_modifiers import bolden_top_k, sort_by_col
from ptls_extension_2024_research.latex_table_creation.prefix_map import get_idxs_where_all_metrics_superpass, prefix_map_from_idx_lst

/path/to/repo/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
with open("results/coles__item_id_embedding_init_with_gnn_embs.txt") as f:
    report_file_content = f.read()

In [10]:
COLES_METRICS = {r"\textbf{AUC test}": 0.877, r"\textbf{ACC test}": 0.793}

In [11]:
from typing import Optional
import re

# Some information is present in name only, everything else is 
# suposed to be retrived from the config (yaml read wit OmegaConf).
# For a uniformal interface both config and experiment_name are always passed to hyperparam getters.


def get_batchsize(config, experiment_name) ->  int:
    return int(config['data_module']['train_batch_size'])


def get_gnn_loss_alpha(config, experiment_name) -> Optional[float]:
    matches = re.search(r'wl-(\d+\.\d+)', experiment_name)
    if matches is None:
        return None
    return float(matches.group(1))


def get_gnn_task(config, experiment_name) -> str:
    if 'GRACE' in experiment_name:
        return 'GRACE'
    return 'Link and weight prediction'


def get_has_residual(config, experiment_name) -> str:
    has_residual_str = re.search(r'residual-(true|True|False)', experiment_name).group(1)
    if  has_residual_str in ['true', 'True']:
        return True
    return False


def get_split_count(config, experiment_name) -> int:
    return int(config['data_module']['train_data']['splitter']['split_count'])


import re

def get_train_epoches(config, experiment_name) -> int:
    # Look for 'epoches_<number>' but exclude 'pretrain_epoches_<number>'
    v1 = re.search(r'(?<!pretrain_)epoches_(\d+)', experiment_name)
    if v1 is not None:
        return int(v1.group(1))
    
    # Look for '<number>_epoches' but exclude 'pretrain_<number>_epoches'
    v2 = re.search(r'(?<!pretrain_)(\d+)_epoches', experiment_name)
    if v2 is not None:
        return int(v2.group(1))
    
    return None

    

hyperparams_to_getters = {
    r"\textbf{GNN Loss alpha}": get_gnn_loss_alpha,
    r"\textbf{GNN task}": get_gnn_task,
    r"\textbf{GNN}": lambda config, experiment_name: re.search(r'gnn-(GraphSAGE|GAT)', experiment_name).group(1),
    r"\textbf{Has residual connection}": get_has_residual,
    r"\textbf{Batch size}": get_batchsize,
    r"\textbf{sc}": get_split_count,
    r"\textbf{weight decay}": lambda config, experiment_name: int(re.search(r'weight_decay-(\d*\.?\d+)', experiment_name).group(1)),
    r"\textbf{Pretrain epoches}": lambda config, experiment_name: int(re.search(r'pretrain_epoches_(\d+)', experiment_name).group(1)),
    r"\textbf{Train epoches}": get_train_epoches,

}

In [12]:
experiment_dicts_list = []

for experiment_name in existing_experiments:
    hydra_config_path = os.path.join('hydra_outputs', experiment_name, '.hydra', 'config.yaml')
    hydra_config_path = re.sub(r'\d+_epoches', 'epoches_150', hydra_config_path)
    config = OmegaConf.load(hydra_config_path)
    hyperparams = {k: v(config, experiment_name) for k, v in hyperparams_to_getters.items()}
    metrics = get_metrics(report_file_content, experiment_name, {'accuracy': r"\textbf{ACC test}"})
    experiment_dicts_list.append({**hyperparams, **metrics})

In [13]:
WEIGHTS_TYPE = 'type 2'

# experiment_dicts_list stores raw values

# experiment_dicts_list stores raw values, list is sorted
experiment_dicts_list = sort_by_col(experiment_dicts_list, r"\textbf{ACC test}")
# prefix_map = prefix_map_from_idx_lst(get_idxs_where_all_metrics_superpass(experiment_dicts_list, COLES_METRICS), "\n\\rowcolor{gray!50}\n")

# experiment_dicts_list stores strings
experiment_dicts_of_str_list = bolden_top_k(experiment_dicts_list, 3, [r"\textbf{ACC test}"])




EXPERIMENT_NAME = r'\makecell{Coles only with bpr}'

experiment_data = {
    EXPERIMENT_NAME: {
        WEIGHTS_TYPE: experiment_dicts_of_str_list
    }
}



In [14]:
hyperparameters = hyperparameter_header_strs = list(experiment_dicts_list[0].keys())

caption = "Coles embedding layer is initialized with pretrained gnn. Scenario age"

row_prefix_dict = None

# row_prefix_dict = {
#     EXPERIMENT_NAME: {
#         WEIGHTS_TYPE: prefix_map
#     }
# }

latex_table = create_latex_table(experiment_data, hyperparameters, hyperparameter_header_strs, caption, row_prefix_dict)

print(latex_table)

\begin{table*}[h!]
\centering
\resizebox{\textwidth}{!}{%
\begin{tabular}{|c|c|c|c|c|c|c|c|c|c|c|c|}
\hline
\textbf{Model} & \textbf{Type of weights} & \textbf{GNN Loss alpha} & \textbf{GNN task} & \textbf{GNN} & \textbf{Has residual connection} & \textbf{Batch size} & \textbf{sc} & \textbf{weight decay} & \textbf{Pretrain epoches} & \textbf{Train epoches} & \textbf{ACC test}\\
\hline
\multirow{49}{*}{\makecell{Coles only with bpr}} &\multirow{49}{*}{type 2} & 0.5 & Link and weight prediction & GAT & True & 40 & 2 & 0 & 91 & 110 & \textbf{0.638} \\ \cline{3-12} 
& &0.5 & Link and weight prediction & GAT & True & 40 & 2 & 0 & 31 & 80 & \textbf{0.636} \\ \cline{3-12} 
& &0.5 & Link and weight prediction & GAT & True & 40 & 2 & 0 & 31 & 100 & \textbf{0.636} \\ \cline{3-12} 
& &0.5 & Link and weight prediction & GAT & True & 40 & 2 & 0 & 91 & 100 & 0.636 \\ \cline{3-12} 
& &0.5 & Link and weight prediction & GraphSAGE & True & 40 & 2 & 0 & 31 & 70 & 0.636 \\ \cline{3-12} 
& &0.5 & Link and